In [ ]:
import dagshub
dagshub.init(repo_owner='tshub23', repo_name='proj1', mlflow=True)

import mlflow

In [ ]:
import pandas as pd


train_df = pd.read_csv("/home/tornike/ML/proj1/train.csv")
test_df = pd.read_csv("/home/tornike/ML/proj1/test.csv")
test_ids = test_df["Id"]

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import mlflow
import mlflow.sklearn
from sklearn.metrics import mean_squared_error
import numpy as np

X = train_df.drop(columns=["SalePrice"])
y = np.log1p(train_df["SalePrice"])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:
na_percent = (train_df.isnull().sum() / len(train_df)) * 100
na_percent = na_percent.sort_values(ascending=False)


high_na_cols = na_percent[na_percent >= 40].index

X_train = X_train.drop(columns=high_na_cols)
X_val = X_val.drop(columns=high_na_cols)

test_df = test_df.drop(columns=high_na_cols)

In [ ]:

categorical_cols = X_train.select_dtypes(include=['object']).columns
print("Categorical columns now:", list(categorical_cols))


for col in categorical_cols:
    
    codes, uniques = pd.factorize(X_train[col].dropna())
    
    
    mapping = dict(zip(uniques, range(len(uniques))))
    
    
    X_train[col] = X_train[col].map(mapping)
    X_val[col]   = X_val[col].map(mapping)
    test_df[col] = test_df[col].map(mapping)
    
    
numeric_cols_with_na = ["LotFrontage", "GarageYrBlt", "MasVnrArea"]

for col in numeric_cols_with_na:
    
    median = X_train[col].median()
    
    X_train[col] = X_train[col].fillna(median)
    X_val[col]   = X_val[col].fillna(median)
    test_df[col] = test_df[col].fillna(median)
    

In [ ]:
# Identify skewed numeric features
numeric_feats = X_train.select_dtypes(include=[np.number]).columns
skewed_feats = X_train[numeric_feats].skew().sort_values(ascending=False)
skewed_feats = skewed_feats[abs(skewed_feats) > 0.75].index

# Log-transform skewed features
for feat in skewed_feats:
    X_train[feat] = np.log1p(X_train[feat])
    X_val[feat]   = np.log1p(X_val[feat])
    test_df[feat] = np.log1p(test_df[feat])

In [ ]:
def drop_high_correlation_features(X_train, X_val, test_df, threshold=0.9):
    corr_matrix = X_train.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    print(f"Dropping {len(to_drop)} columns due to correlation > {threshold}: {to_drop}")
    X_train_filtered = X_train.drop(columns=to_drop)
    X_val_filtered   = X_val.drop(columns=to_drop)
    test_df_filtered = test_df.drop(columns=to_drop)
    return X_train_filtered, X_val_filtered, test_df_filtered

X_train, X_val, test_df = drop_high_correlation_features(X_train, X_val, test_df, threshold=0.8)

In [ ]:

quality_mapping = {"Ex":5, "Gd":4, "TA":3, "Fa":2, "Po":1}

for col in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "KitchenQual", "GarageQual", "GarageCond"]:
    X_train[col] = X_train[col].map(quality_mapping)
    X_val[col]   = X_val[col].map(quality_mapping)
    test_df[col] = test_df[col].map(quality_mapping)


for col in X_train.select_dtypes(include='object').columns:
    X_train[col] = X_train[col].fillna("None")
    X_val[col]   = X_val[col].fillna("None")
    test_df[col] = test_df[col].fillna("None")

# One-hot encode nominal categoricals
X_train = pd.get_dummies(X_train)
X_val   = pd.get_dummies(X_val)
test_df = pd.get_dummies(test_df)

# Align columns
X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)
X_train, test_df = X_train.align(test_df, join='left', axis=1, fill_value=0)

In [ ]:
with mlflow.start_run(run_name="removed rfe with 21 features"):
    
    import xgboost as xgb
    from sklearn.metrics import mean_squared_error
    import numpy as np

    model = xgb.XGBRegressor(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )

    model.fit(X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=50)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    print("RMSE with XGBoost:", rmse)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, "model")
    
   
    test_preds = np.expm1(model.predict(test_df))

    submission = pd.DataFrame({
        "Id": test_ids,
        "SalePrice": test_preds
    })
    submission.to_csv("submission.csv", index=False)
    
